In [1]:
#!/usr/bin/env python3

import json
import subprocess
import sys
from pathlib import Path


# ============================================================
# Configuration
# ============================================================

VTM_ROOT = Path("/home/yuanzn/Documents/QUIC_scheduling/VVCSoftware_VTM")

VTM_BIN = (
    VTM_ROOT
    / "bin/umake/gcc-13.3/x86_64/release"
)

EXTRACTOR = VTM_BIN / "BitstreamExtractorApp"

NUM_SUBPICS = 2


# ============================================================
# VVC NAL types
#
# VCL NAL units: 0 ~ 11
#
# Non-VCL:
# 12 OPI
# 13 DCI
# 14 VPS
# 15 SPS
# 16 PPS
# 17 PREFIX_APS
# 18 SUFFIX_APS
# 19 PH
# 20 AUD
# 21 EOS
# 22 EOB
# 23 PREFIX_SEI
# 24 SUFFIX_SEI
# 25 FD
# ============================================================

NAL_NAMES = {
    0:  "TRAIL",
    1:  "STSA",
    2:  "RADL",
    3:  "RASL",
    7:  "IDR_W_RADL",
    8:  "IDR_N_LP",
    9:  "CRA",
    10: "GDR",

    12: "OPI",
    13: "DCI",
    14: "VPS",
    15: "SPS",
    16: "PPS",
    17: "PREFIX_APS",
    18: "SUFFIX_APS",
    19: "PH",
    20: "AUD",
    21: "EOS",
    22: "EOB",
    23: "PREFIX_SEI",
    24: "SUFFIX_SEI",
    25: "FD",
}


# ============================================================
# Annex-B parser
# ============================================================

def find_start_codes(data):
    """
    Return [(position, start_code_length), ...]
    Supports both:
        00 00 01
        00 00 00 01
    """

    result = []

    i = 0

    while i < len(data) - 3:

        if data[i:i+4] == b"\x00\x00\x00\x01":
            result.append((i, 4))
            i += 4
            continue

        if data[i:i+3] == b"\x00\x00\x01":
            result.append((i, 3))
            i += 3
            continue

        i += 1

    return result


def parse_annexb(path):

    data = Path(path).read_bytes()

    starts = find_start_codes(data)

    if not starts:
        raise RuntimeError(
            f"No Annex-B start code found in {path}"
        )

    nalus = []

    for i, (start, sc_len) in enumerate(starts):

        if i + 1 < len(starts):
            end = starts[i + 1][0]
        else:
            end = len(data)

        raw = data[start:end]

        payload_start = start + sc_len

        # VVC NAL header = 2 bytes
        if payload_start + 2 > end:
            continue

        header = data[payload_start:payload_start + 2]

        # VVC:
        # second byte:
        #
        # nal_unit_type       = bits 7..3
        # nuh_temporal_id+1   = bits 2..0
        #
        nal_type = (header[1] >> 3) & 0x1F

        nalus.append({
            "index": len(nalus),
            "type": nal_type,
            "name": NAL_NAMES.get(
                nal_type,
                f"NAL_{nal_type}"
            ),
            "raw": raw,
        })

    return nalus


# ============================================================
# Extract one subpicture using official VTM tool
# ============================================================

def run_extractor(input_vvc, output_vvc, subpic_idx):

    cmd = [
        str(EXTRACTOR),

        "-b",
        str(input_vvc),

        "-o",
        str(output_vvc),

        f"--SubPicIdx={subpic_idx}",
    ]

    print()
    print("Running:")
    print(" ".join(cmd))
    print()

    result = subprocess.run(cmd)

    if result.returncode != 0:
        raise RuntimeError(
            f"BitstreamExtractorApp failed for "
            f"SubPicIdx={subpic_idx}"
        )


# ============================================================
# Get only VCL NALUs
# ============================================================

def get_vcl_nalus(nalus):
    return [
        n
        for n in nalus
        if 0 <= n["type"] <= 11
    ]


# ============================================================
# Build a fingerprint for matching NALUs
# ============================================================

def nal_payload_without_start_code(raw):

    if raw.startswith(b"\x00\x00\x00\x01"):
        return raw[4:]

    if raw.startswith(b"\x00\x00\x01"):
        return raw[3:]

    return raw


# ============================================================
# Main split operation
# ============================================================

def split(input_vvc, output_dir):

    input_vvc = Path(input_vvc)
    output_dir = Path(output_dir)

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    print("=" * 70)
    print("Input:", input_vvc)
    print("=" * 70)

    # --------------------------------------------------------
    # Parse original VVC
    # --------------------------------------------------------

    original_nalus = parse_annexb(
        input_vvc
    )

    print()
    print("Original NAL units:")
    print()

    for n in original_nalus:

        print(
            f'{n["index"]:4d} '
            f'type={n["type"]:2d} '
            f'{n["name"]:15s} '
            f'{len(n["raw"]):8d} bytes'
        )

    # --------------------------------------------------------
    # Use BitstreamExtractorApp to determine which original
    # VCL NAL belongs to which subpicture
    # --------------------------------------------------------

    subpic_payload_sets = []

    for sid in range(NUM_SUBPICS):

        tmp_file = (
            output_dir
            / f"_extractor_subpic_{sid}.vvc"
        )

        run_extractor(
            input_vvc,
            tmp_file,
            sid,
        )

        extracted_nalus = parse_annexb(
            tmp_file
        )

        vcl = get_vcl_nalus(
            extracted_nalus
        )

        payloads = set()

        for n in vcl:

            payloads.add(
                nal_payload_without_start_code(
                    n["raw"]
                )
            )

        subpic_payload_sets.append(
            payloads
        )

        print(
            f"Subpic {sid}: "
            f"{len(vcl)} VCL NAL units"
        )

    # --------------------------------------------------------
    # Classify ORIGINAL NAL units
    #
    # This is important:
    #
    # We save bytes from original.vvc,
    # NOT bytes rewritten by BitstreamExtractorApp.
    #
    # Therefore merge can reconstruct original stream exactly.
    # --------------------------------------------------------

    manifest = []

    common_file = (
        output_dir / "common.vvc"
    )

    subpic_files = [
        output_dir / f"subpic_{sid}.vvc"
        for sid in range(NUM_SUBPICS)
    ]

    common_fp = open(
        common_file,
        "wb"
    )

    subpic_fps = [
        open(p, "wb")
        for p in subpic_files
    ]

    common_idx = 0
    subpic_idx = [0] * NUM_SUBPICS

    try:

        for n in original_nalus:

            raw = n["raw"]

            # --------------------------------------------
            # Non-VCL -> COMMON
            # --------------------------------------------

            if not (0 <= n["type"] <= 11):

                common_fp.write(raw)

                manifest.append({
                    "source": "common",
                    "local_index": common_idx,
                    "nal_type": n["type"],
                    "nal_name": n["name"],
                    "size": len(raw),
                })

                common_idx += 1

                continue

            # --------------------------------------------
            # VCL -> determine Subpicture
            # --------------------------------------------

            payload = (
                nal_payload_without_start_code(
                    raw
                )
            )

            owner = None

            for sid in range(NUM_SUBPICS):

                if payload in subpic_payload_sets[sid]:

                    owner = sid
                    break

            if owner is None:

                raise RuntimeError(
                    "Could not determine subpicture "
                    f"for original NAL #{n['index']} "
                    f"type={n['type']}."
                )

            subpic_fps[owner].write(
                raw
            )

            manifest.append({
                "source": f"subpic_{owner}",
                "local_index":
                    subpic_idx[owner],

                "nal_type": n["type"],
                "nal_name": n["name"],
                "size": len(raw),
            })

            subpic_idx[owner] += 1

    finally:

        common_fp.close()

        for fp in subpic_fps:
            fp.close()

    # --------------------------------------------------------
    # Save manifest
    # --------------------------------------------------------

    manifest_path = (
        output_dir / "manifest.json"
    )

    with open(
        manifest_path,
        "w"
    ) as f:

        json.dump(
            manifest,
            f,
            indent=2
        )

    print()
    print("=" * 70)
    print("Split completed")
    print("=" * 70)

    print(
        "Common:",
        common_file,
        common_file.stat().st_size,
        "bytes"
    )

    for sid, p in enumerate(
        subpic_files
    ):

        print(
            f"Subpic {sid}:",
            p,
            p.stat().st_size,
            "bytes"
        )

    print(
        "Manifest:",
        manifest_path
    )


# ============================================================
# Read individual NALUs from a split file
# ============================================================

def load_split_nalus(path):

    if not Path(path).exists():
        raise RuntimeError(
            f"Missing file: {path}"
        )

    return parse_annexb(path)


# ============================================================
# Merge
# ============================================================

def merge(input_dir, output_vvc):

    input_dir = Path(input_dir)

    manifest_path = (
        input_dir / "manifest.json"
    )

    manifest = json.loads(
        manifest_path.read_text()
    )

    # Load all individual streams

    streams = {}

    streams["common"] = (
        load_split_nalus(
            input_dir / "common.vvc"
        )
    )

    for sid in range(NUM_SUBPICS):

        streams[f"subpic_{sid}"] = (
            load_split_nalus(
                input_dir
                / f"subpic_{sid}.vvc"
            )
        )

    output_vvc = Path(output_vvc)

    with open(
        output_vvc,
        "wb"
    ) as out:

        for item in manifest:

            source = item["source"]
            local_index = item[
                "local_index"
            ]

            nalu = streams[
                source
            ][local_index]

            out.write(
                nalu["raw"]
            )

    print()
    print("=" * 70)
    print("Merge completed")
    print("=" * 70)

    print(
        "Output:",
        output_vvc,
        output_vvc.stat().st_size,
        "bytes"
    )


def main():

    # ============================================================
    # Experiment configuration
    # ============================================================

    MODE = "split"
    # MODE = "merge"

    # Original VVC bitstream
    INPUT_VVC = (
        "/home/yuanzn/Documents/QUIC_scheduling/VVCSoftware_VTM/experiments/subpic_official/official_2subpic.vvc"
    )

    # Directory containing split bitstreams
    OUTPUT_DIR = (
        "subpic_split"
    )

    # Output after merging
    MERGED_VVC = (
        "subpic_split/"
        "merged.vvc"
    )

    # ============================================================
    # Run
    # ============================================================

    if MODE == "split":

        print("=" * 70)
        print("VVC Subpicture Split")
        print("=" * 70)

        print(f"Input VVC : {INPUT_VVC}")
        print(f"Output dir: {OUTPUT_DIR}")

        split(
            INPUT_VVC,
            OUTPUT_DIR
        )

    elif MODE == "merge":

        print("=" * 70)
        print("VVC Subpicture Merge")
        print("=" * 70)

        print(f"Input dir : {OUTPUT_DIR}")
        print(f"Output VVC: {MERGED_VVC}")

        merge(
            OUTPUT_DIR,
            MERGED_VVC
        )

    else:

        raise ValueError(
            f"Unknown MODE: {MODE}"
        )


if __name__ == "__main__":
    main()

VVC Subpicture Split
Input VVC : /home/yuanzn/Documents/QUIC_scheduling/VVCSoftware_VTM/experiments/subpic_official/official_2subpic.vvc
Output dir: subpic_split
Input: /home/yuanzn/Documents/QUIC_scheduling/VVCSoftware_VTM/experiments/subpic_official/official_2subpic.vvc

Original NAL units:

   0 type=15 SPS                   44 bytes
   1 type=16 PPS                   35 bytes
   2 type=17 PREFIX_APS            16 bytes
   3 type=17 PREFIX_APS            97 bytes
   4 type=19 PH                     8 bytes
   5 type= 8 IDR_N_LP            2010 bytes
   6 type= 8 IDR_N_LP             912 bytes
   7 type= 8 IDR_N_LP            1666 bytes
   8 type= 8 IDR_N_LP             743 bytes
   9 type= 8 IDR_N_LP           12891 bytes
  10 type= 8 IDR_N_LP            4496 bytes
  11 type= 8 IDR_N_LP            5065 bytes
  12 type= 8 IDR_N_LP            1505 bytes

Running:
/home/yuanzn/Documents/QUIC_scheduling/VVCSoftware_VTM/bin/umake/gcc-13.3/x86_64/release/BitstreamExtractorApp -b /home/yua